# JSON
- JSON은 데이터를 사람이 읽을 수 있는 텍스트로 표현하면서, 컴퓨터가 쉽게 파싱할 수 있는 경량 데이터 교환 형식이다.
- 웹에서 서버와 클라이언트 간 데이터 교환에 널리 사용한다. 
- javaScript 객체 문법을 기반으로 만들어졌지만, python, java, c 등 거의 모든 언어에서 사용 가능하다. 

# JSONLoader
> JSONLoader은 JSON 파일을 읽어서 각 항목을 Document 객체로 변환해주는 문서 로더이다. 
> 즉, JSON 데이터의 특정 필드를 page_content로 지정해서 LLM이 이해할 수 있는 텍스트 문서 형태로 바꿔주는 역할을 한다.

In [1]:
# sokcho_travel_guide.json

# 데이터 확인 

import json 

file_path = "./data/sokcho_travel_guide.json"
data = json.load(open(file_path,"r",encoding="utf-8"))

In [2]:
from pprint import pprint

pprint(data)

{'date': '2025-07-15',
 'local_cuisine': [{'description': '속초의 명물! 달콤하고 바삭한 맛이 특징', 'dish': '닭강정'},
                   {'description': '오징어 안에 찹쌀과 채소를 넣어 찐 전통 음식',
                    'dish': '오징어순대'}],
 'location': '강원특별자치도 속초시',
 'tourist_attractions': [{'description': '아름다운 해변과 시원한 바닷바람이 매력적인 명소',
                          'name': '속초해수욕장',
                          'tip': '여름철 피서지로 인기 많음'},
                         {'description': '울산바위, 권금성 등 다양한 명소가 있는 국립공원',
                          'name': '설악산 국립공원',
                          'tip': '트레킹과 자연 풍경 감상에 최적'},
                         {'description': '6.25 전쟁 당시 피난민이 정착한 전통 마을',
                          'name': '아바이마을',
                          'tip': '갯배 체험과 향토 음식 즐기기 가능'}],
 'travel_tips': ['해수욕장과 산을 함께 즐길 수 있는 복합형 여행지',
                 '중앙시장 등지에서 지역 음식을 현지 스타일로 맛볼 수 있음',
                 '아바이마을 갯배 체험은 필수 코스'],
 'weather': {'condition': '맑음', 'humidity': '60%', 'temperature': '25°C'}}


In [ ]:
from langchain_community.document_loaders import JSONLoader

# AI가 먹을 수 있는 형태(Document)로 JSON 데이터를 변환해 주는 로더(도구)를 세팅합니다.
loader = JSONLoader(
    # 1. 재료 위치: 읽어올 JSON 파일의 경로를 지정합니다.
    file_path=file_path,
    
    # 2. 작업 지시서 (jq_schema): 어떤 데이터를 뽑아서 어떻게 조립할지 정해줍니다.
    # r'...' : 파이썬에서 특수문자(\)를 오류 없이 그대로 쓰기 위한 표시(Raw string)입니다.
    # .tourist_attractions[] : "tourist_attractions 상자를 열어서 안에 있는 걸 하나씩 꺼내라!"
    # | : "꺼낸 것을 다음 포장 단계로 넘겨라! (컨베이어 벨트)"
    # "\(...)" : "이름표에 적힌 알맹이만 빼와서, '이름:설명(팁:팁내용)' 이라는 하나의 문장으로 예쁘게 포장해라!"
    jq_schema=r'.tourist_attractions[]|"\(.name):(.description)(팁:\(.tip))"',
    
    # 3. 텍스트 추출 방식: 우리가 JSON 안의 단순한 텍스트 값 하나만 가져오는 게 아니라, 
    # 위에서 복잡하게 문자열을 새로 조립했기 때문에 False로 설정해 줍니다.
    text_content=False
)

# 4. 스위치 ON: 세팅된 로더를 실제로 작동시켜서 데이터를 읽어오고 조립한 뒤, 
# 'documents'라는 리스트(접시)에 차곡차곡 담아줍니다.
documents = loader.load()

In [5]:
print(f"로드된 파일의 수:{len(documents)}")

로드된 파일의 수:3


In [6]:
for i, doc in enumerate(documents):
    print("="*50)
    print(f"Document {i+1}:")
    print(f"   - 타입: {type(doc)}")
    print(f"   - page_content 타입: {type(doc.page_content)}")
    print(f"   - metadata 타입: {type(doc.metadata)}")
    print(f"   - metadata 내용: {doc.metadata}")
    print(f"   - 내용 길이: {len(doc.page_content)} 문자")
    print(f"   - 내용: {doc.page_content}")


Document 1:
   - 타입: <class 'langchain_core.documents.base.Document'>
   - page_content 타입: <class 'str'>
   - metadata 타입: <class 'dict'>
   - metadata 내용: {'source': 'C:\\dev\\study\\LLM\\RAG\\1. Naive RAG\\ex01\\data\\sokcho_travel_guide.json', 'seq_num': 1}
   - 내용 길이: 39 문자
   - 내용: 속초해수욕장:(.description)(팁:여름철 피서지로 인기 많음)
Document 2:
   - 타입: <class 'langchain_core.documents.base.Document'>
   - page_content 타입: <class 'str'>
   - metadata 타입: <class 'dict'>
   - metadata 내용: {'source': 'C:\\dev\\study\\LLM\\RAG\\1. Naive RAG\\ex01\\data\\sokcho_travel_guide.json', 'seq_num': 2}
   - 내용 길이: 44 문자
   - 내용: 설악산 국립공원:(.description)(팁:트레킹과 자연 풍경 감상에 최적)
Document 3:
   - 타입: <class 'langchain_core.documents.base.Document'>
   - page_content 타입: <class 'str'>
   - metadata 타입: <class 'dict'>
   - metadata 내용: {'source': 'C:\\dev\\study\\LLM\\RAG\\1. Naive RAG\\ex01\\data\\sokcho_travel_guide.json', 'seq_num': 3}
   - 내용 길이: 43 문자
   - 내용: 아바이마을:(.description)(팁:갯배 체험과 향토 음식 즐기기 가능)


In [13]:
from langchain_community.document_loaders import JSONLoader

# JSON에서 관광지 이름만 추출하는 로더 설정
loader_name = JSONLoader(
    file_path=file_path,                      # 대상 JSON 파일 경로
    jq_schema='.tourist_attractions[].name',  # 추출할 데이터 위치 (이름만 쏙)
    text_content=False                        # 원본 데이터 구조 유지
)

# 로더를 실행하여 문서(Document) 리스트로 저장
name_docs = loader_name.load()

In [14]:
print(f"로드된 파일의 수:{len(name_docs)}")

로드된 파일의 수:3


In [16]:
print("==관광지 이름 ===")

# 문서(doc)와 순서 번호(i)를 하나씩 꺼냄
for i, doc in enumerate(name_docs):
    # 번호(1부터 시작)와 문서의 핵심 텍스트(page_content)를 출력
    print(f"{i+1}.{doc.page_content}")

==관광지 이름 ===
1.속초해수욕장
2.설악산 국립공원
3.아바이마을


In [ ]:
# 💡 JSON 원본 데이터에서 특정 값을 뽑아 커스텀 메타데이터(꼬리표)를 만드는 함수

def metadata_func(record: dict, metadata: dict) -> dict:
    # 'local_cuision' 목록에서 'dish'(요리 이름)만 뽑아 "travel_foods"라는 새 메타데이터로 저장
    metadata["travel_foods"] = [food["dish"] for food in record.get("local_cuision", [])]
    
    # 내용이 추가된 메타데이터 반환
    return metadata

In [ ]:
# 💡 [한 줄 요약]
# JSON 파일 전체(.)를 통째로 가져오면서, 앞서 만든 맞춤형 꼬리표(metadata_func)까지 붙여 AI 문서로 만드는 코드

loader_desc = JSONLoader(
    file_path=file_path,            # 1. 대상 JSON 파일 경로
    jq_schema='.',                  # 2. 추출 위치: '.'은 "JSON 상자 전체를 통째로 가져와라"라는 뜻
    text_content=False,             # 3. 데이터 형식: 원본 딕셔너리 구조 유지 (자동으로 문자열 변환됨)
    metadata_func=metadata_func     # 4. 맞춤형 꼬리표: 아까 만든 함수를 연결해 'travel_foods' 정보 추가
)

# 5. 스위치 ON: 설정한 대로 데이터를 읽어와 문서(Document) 리스트로 완성
docs = loader_desc.load()

In [ ]:
print(f"로드된 파일의 수: {len(docs)}")

In [ ]:
for i, doc in enumerate(docs):
    print("="*70)
    print(f"Document {i+1}:")
    print(f"   - 타입: {type(doc)}")
    print(f"   - page_content 타입: {type(doc.page_content)}")
    print(f"   - metadata 타입: {type(doc.metadata)}")
    print(f"   - metadata 내용(travel_foods): {doc.metadata['travel_foods']}")
    print(f"   - metadata 전체 내용: {doc.metadata}")
    print(f"   - 내용 길이: {len(doc.page_content)} 문자")
    # JSON 문자열을 파싱한 후 ensure_ascii=False로 다시 출력하여 한글 표시
    import json
    content_dict = json.loads(doc.page_content)
    print(f"   - 내용 (한글 표시):")
    print(json.dumps(content_dict, ensure_ascii=False, indent=2))
    print()